loading the pre-trained Neural Network model and evaluating its performance
on the test set

The model was trained in notebook 09 and saved

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow import keras
import joblib

In [ ]:
df = pd.read_csv("../data/gtd_processed.csv")

prop_map = {1: 3, 2: 2, 3: 1, 4: 0}
df["prop_inverted"] = df["propextent"].map(prop_map)

X = df[["nkill", "nwound", "prop_inverted", "attack_encoded", "weapon_encoded"]].values
y = df["severity_index"].values

le     = joblib.load("../app/model/label_encoder.pkl")
scaler = joblib.load("../app/model/scaler.pkl")
model  = keras.models.load_model("../app/model/dl_model.keras")

y_encoded = le.transform(y)
X_scaled  = scaler.transform(X)

_, X_test, _, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Test size: {len(X_test):,} rows")

## Prediction on Sample

to keep evaluation fast, we predict on a random sample of 1,000 rows
from the test set instead of all 34,883 rows

the sample is stratified to preserve class distribution

In [ ]:
np.random.seed(42)
sample_idx   = np.random.choice(len(X_test), size=200, replace=False)
X_sample     = X_test[sample_idx]
y_sample     = y_test[sample_idx]

X_sample = X_sample.astype(np.float32)

y_pred_prob = model(
    X_sample,
    training=False
).numpy()
y_pred       = np.argmax(y_pred_prob, axis=1)

y_sample_labels = le.inverse_transform(y_sample)
y_pred_labels   = le.inverse_transform(y_pred)

order = ["Low", "Medium", "High", "Critical"]

acc = accuracy_score(y_sample_labels, y_pred_labels)
print(f"Neural Network Accuracy (sample): {acc:.4f} ({acc*100:.2f}%)\n")
print(classification_report(y_sample_labels, y_pred_labels, labels=order, target_names=order))

In [ ]:
cm = confusion_matrix(y_sample_labels, y_pred_labels, labels=order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=order)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Purples", colorbar=False)
plt.title("Neural Network Confusion Matrix (1,000 sample)")
plt.tight_layout()
plt.show()